# 03 — Build donor × region pseudobulk

Computes and writes the pseudobulk data frames consumed by `04_progression.ipynb`
and `05_mixed_models.ipynb`.  Run this once after training or whenever the embedding
changes; all downstream notebooks load from the parquet files written here.

| Output | Shape | Used by |
|--------|-------|---------|
| `activity.parquet` | donor×region × live directions | `04_progression` |
| `meta.parquet` | donor×region × (CPS + n_cells) | `04_progression` |
| `within_activity.parquet` | donor×region×supertype × live directions | `04_progression` |
| `within_meta.parquet` | donor×region×supertype × (CPS_Local + n_cells) | `04_progression` |
| `dat.parquet` | donor×region × (factor_1..128 + covariates) | `05_mixed_models` |

In [ ]:
from __future__ import annotations

import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp

import sst_drvi as S

## Embed

In [ ]:
CANDIDATES = ["sst_k64_covar_long", "sst_k64_covar", "sst_k64"]
MODEL = "drvi"

RUN = None
for candidate in CANDIDATES:
    embed_path = S.EMBED_DIR / f"{candidate}_{MODEL}_embed.h5ad"
    if embed_path.exists():
        RUN = candidate
        break
    print(f"{candidate} not on disk")
if RUN is None:
    raise FileNotFoundError(f"no DRVI embedding found among {CANDIDATES}")

embed = ad.read_h5ad(embed_path)
SUPERTYPES = list(embed.obs["Supertype"].cat.categories)

print(f"run: {RUN}")
print(f"{embed.n_obs:,} nuclei x {embed.n_vars} factors, "
      f"{len(SUPERTYPES)} supertypes, {embed.obs['Donor ID'].nunique()} donors")

## Helpers

`split_by_sign` splits each latent dimension into two non-negative ReLU columns.
`pseudobulk_directions` uses it to aggregate per-cell activations to group means.

In [ ]:
# `logger` is wired up because both helpers report what they drop -- units under the
# cell floor, directions flagged as vanished -- and those messages are worth seeing.
import logging
import warnings
from collections.abc import Sequence

logger = logging.getLogger("pseudobulk")
if not logger.handlers:
    logger.addHandler(logging.StreamHandler())
    logger.setLevel(logging.INFO)


def split_by_sign(embed, hide_vanished: bool = True) -> pd.DataFrame:
    """Per-cell activation magnitude of each factor *direction*.

    DRVI factors are directional: ``DR n+`` and ``DR n-`` can encode unrelated programs,
    which is why the interpretability scores are computed per direction. A signed factor
    value conflates the two, so an association carried by only one direction is diluted
    by the cells sitting on the other side of zero. This returns ``relu(x)`` and
    ``relu(-x)`` as separate non-negative columns.

    Columns are labelled and ordered exactly like `interpretability_scores`
    (``DR 1+``, ``DR 1-``, ``DR 2+``, ...), so the two tables join on them. Directions
    flagged vanished by ``set_latent_dimension_stats`` are dropped by default.
    """
    x = np.asarray(embed.X, dtype=np.float32)
    info = embed.var
    columns = {}
    for pos, (_, row) in enumerate(info.iterrows()):
        values = x[:, pos]
        for direction, magnitude, dead in (
            ("+", np.maximum(values, 0.0), row["vanished_positive_direction"]),
            ("-", np.maximum(-values, 0.0), row["vanished_negative_direction"]),
        ):
            if hide_vanished and dead:
                continue
            columns[f"{row['title']}{direction}"] = magnitude

    out = pd.DataFrame(columns, index=embed.obs_names)
    order = (
        pd.concat([info.assign(direction="+"), info.assign(direction="-")])
        .assign(title=lambda df: df["title"] + df["direction"])
        .sort_values(["order", "direction"])["title"]
    )
    return out[[c for c in order if c in out.columns]]


def pseudobulk_directions(
    embed,
    keys: Sequence[str] = ("Donor ID", "Brain Region"),
    hide_vanished: bool = True,
    min_cells: int = 20,
    carry: Sequence[str] = (),
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Mean activity of each factor *direction* per unit of analysis.

    Returns ``(activity, meta)`` sharing one index: ``activity`` is units x directions
    (mean of the non-negative ``relu`` magnitudes from `split_by_sign`), ``meta``
    carries ``n_cells`` plus the first value of every column in ``carry``.

    The mean is over cells, so a unit built from 40 nuclei is noisier than one built from
    4,000 and ``min_cells`` drops the worst of them; ``n_cells`` is returned so the rest
    can be weighted or inspected rather than silently trusted.
    """
    split = split_by_sign(embed, hide_vanished=hide_vanished)
    obs = embed.obs
    keys = list(keys)
    missing = [k for k in [*keys, *carry] if k not in obs]
    if missing:
        raise KeyError(f"obs is missing {missing}")

    grouper = [obs[k] for k in keys]
    activity = split.groupby(grouper, observed=True).mean()
    meta = pd.DataFrame({"n_cells": obs.groupby(grouper, observed=True).size()})
    if carry:
        meta = meta.join(obs.groupby(grouper, observed=True)[list(carry)].first())

    keep = meta["n_cells"] >= min_cells
    if not keep.all():
        logger.info(
            "dropping %d of %d %s units with fewer than %d cells",
            (~keep).sum(),
            len(keep),
            " x ".join(keys),
            min_cells,
        )
    activity, meta = activity[keep.to_numpy()], meta[keep]

    # A score that varies inside a unit was aggregated at the wrong level; say so rather
    # than averaging it away, since the whole point of the unit is that it is constant.
    for col in carry:
        if col in S.CPS_COLUMNS:
            spread = (
                pd.to_numeric(obs[col], errors="coerce")
                .groupby(grouper, observed=True)
                .nunique()
                .max()
            )
            if spread > 1:
                logger.warning(
                    "%s takes up to %d values within a single %s unit; "
                    "meta holds only the first",
                    col,
                    spread,
                    " x ".join(keys),
                )
    return activity, meta

## Donor × region pseudobulk

Mean ReLU direction activations per donor × brain-region unit.  `carry` columns
must be constant within each unit; CPS scores satisfy this at the donor × region level.

In [ ]:
activity, meta = pseudobulk_directions(
    embed, keys=("Donor ID", "Brain Region"),
    carry=S.CPS_COLUMNS + ("Cognitive Status", "Braak", "Sex", "Age at Death", "PMI"),
)
print(f"{activity.shape[0]} donor x region units x {activity.shape[1]} live directions")
print(f"median nuclei per unit: {meta['n_cells'].median():.0f} "
      f"(min {meta['n_cells'].min()}, max {meta['n_cells'].max()})")

## Donor × region × supertype pseudobulk

Finer units for the composition-free test in `04_progression` section 3.  The
20-cell floor removes sparse supertype × region × donor combinations.

In [ ]:
within, within_meta = pseudobulk_directions(
    embed, keys=("Donor ID", "Brain Region", "Supertype"),
    min_cells=20, carry=("CPS_Local",),
)
print(f"{len(within)} donor x region x supertype units, "
      f"{within_meta.index.get_level_values('Supertype').nunique()} supertypes")

## Factor activations for 05_mixed_models

`05_mixed_models` tests each of the 128 ReLU-split factor directions against CPS
phenotypes with donor-random-intercept linear models.  The unit of analysis is the
same donor × region pseudobulk, but summarised as mean activation with raw factor
columns rather than the labelled directional split from `pseudobulk_directions`.

In [ ]:
X = embed.X.toarray() if sp.issparse(embed.X) else np.asarray(embed.X)
n_dims = X.shape[1]
print(f"Latent dimensions: {n_dims}")

obs = embed.obs[
    ["Donor ID", "Brain Region", "Sex", "Age at Death", "PMI", *S.CPS_COLUMNS]
].rename(columns={
    "Donor ID": "donor",
    "Brain Region": "region",
    "Sex": "sex",
    "Age at Death": "age",
})

factor_data = {}
for i in range(n_dims):
    factor_data[f"factor_{2 * i + 1}"] = np.maximum(X[:, i], 0.0)
    factor_data[f"factor_{2 * i + 2}"] = np.maximum(-X[:, i], 0.0)
factor_cols = list(factor_data)

obs = pd.concat([obs, pd.DataFrame(factor_data, index=obs.index)], axis=1)

agg_keys = {f: "mean" for f in factor_cols}
agg_keys |= {c: "first" for c in [*S.CPS_COLUMNS, "sex", "age", "PMI"]}

dat = obs.groupby(["donor", "region"]).agg(agg_keys).reset_index()
print(
    f"Pseudobulk: {len(dat):,} donor×region units, "
    f"{dat['donor'].nunique()} donors, {dat['region'].nunique()} regions"
)

## Write

In [ ]:
PSEUDO_DIR = S.SCRATCH / "pseudobulk"
PSEUDO_DIR.mkdir(parents=True, exist_ok=True)

activity.to_parquet(PSEUDO_DIR / "activity.parquet")
meta.to_parquet(PSEUDO_DIR / "meta.parquet")
within.to_parquet(PSEUDO_DIR / "within_activity.parquet")
within_meta.to_parquet(PSEUDO_DIR / "within_meta.parquet")
dat.to_parquet(PSEUDO_DIR / "dat.parquet", index=False)

print(f"Written to {PSEUDO_DIR}:")
for f in sorted(PSEUDO_DIR.glob("*.parquet")):
    sz = f.stat().st_size / 1024
    print(f"  {f.name}  ({sz:.0f} KB)")